# Classificatore basato su SVM lineari e n-grammi.

## In questo notebook si svolge il primo punto del progetto, ossia: classificatore svm lineari su n-grams, parole e pos. Sono state sperimentate le seguenti configurazioni: unigrammi, bigrammi, trigrammi di parola, lemma e pos, trigrammi, quadrigrammi e 5-grams di carattere. Tramite la cross validation a 5 fold il modello migliore individuato è unigrammi di parola.

# 1) Import

In [47]:
import stanza
stanza.download('it')

2026-06-04 15:49:58 INFO: Downloaded file to C:\Users\jacop\AppData\Local\StanfordNLP\stanza\Cache\1.12.0\resources\resources.json
2026-06-04 15:49:58 INFO: Downloading default packages for language: it (Italian) ...
2026-06-04 15:49:59 INFO: File exists: C:\Users\jacop\AppData\Local\StanfordNLP\stanza\Cache\1.12.0\resources\it\default.zip
2026-06-04 15:50:02 INFO: Finished downloading models and saved to C:\Users\jacop\AppData\Local\StanfordNLP\stanza\Cache\1.12.0\resources


[['zip', 'default.zip']]

In [48]:
import pandas as pd
import matplotlib.pyplot as plt

In [49]:
from scipy.stats import mannwhitneyu

In [50]:
import importlib
import Classes
importlib.reload(Classes)
from Classes import *

In [51]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score

In [52]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

In [53]:
nlp = stanza.Pipeline('it', processors='tokenize,mwt,pos,lemma')

2026-06-04 15:50:02 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-06-04 15:50:02 INFO: Downloaded file to C:\Users\jacop\AppData\Local\StanfordNLP\stanza\Cache\1.12.0\resources\resources.json
2026-06-04 15:50:03 INFO: Loading these models for language: it (Italian):
| Processor | Package           |
---------------------------------
| tokenize  | combined_nocharlm |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2026-06-04 15:50:03 INFO: Using device: cpu
2026-06-04 15:50:03 INFO: Loading: tokenize
2026-06-04 15:50:03 INFO: Loading: mwt
2026-06-04 15:50:03 INFO: Loading: pos
2026-06-04 15:50:04 INFO: Loading: lemma
2026-06-04 15:50:05 INFO: Done loading processors!


## 2) Caricamento datatset e annotazione

In [54]:
df_train = pd.read_csv("df_train.csv")
df_train

,Unnamed: 0,text,label
0,9610,Il dato emerge dall’ultima pubblicazione dell’...,0
1,29392,Ora si fanno i conti. Ecco allora tutti i Paes...,0
2,24989,"Lo stesso purtroppo non si può dire per la "" f...",0
3,14323,"Ma oggi il materiale diffuso sul web, che most...",0
4,5346,"Gli specialisti del Gis, peraltro in continuo ...",0
...,...,...,...
1995,26090,"Si dice che l'uomo, di età compresa tra i 38 e...",1
1996,9218,"La strage di Detroit, con cinque innocenti mas...",1
1997,25409,L’omicidio è avvenuto in un’abitazione di piaz...,1
1998,27653,"I dispositivi Android, spesso acquistati da fa...",1


In [55]:
"train_documents = get_data_ann(df_train, nlp)"

'train_documents = get_data_ann(df_train, nlp)'

In [56]:
"""import pickle
with open("train_documents.pkl", "wb") as f:
    pickle.dump(train_documents, f)"""

'import pickle\nwith open("train_documents.pkl", "wb") as f:\n    pickle.dump(train_documents, f)'

In [57]:
"""df_val = pd.read_csv("df_val.csv")
df_test = pd.read_csv("df_test.csv")"""

'df_val = pd.read_csv("df_val.csv")\ndf_test = pd.read_csv("df_test.csv")'

In [58]:
"""val_documents = get_data_ann(df_val,nlp)
with open("val_documents.pkl", "wb") as f:
    pickle.dump(val_documents, f)"""

'val_documents = get_data_ann(df_val,nlp)\nwith open("val_documents.pkl", "wb") as f:\n    pickle.dump(val_documents, f)'

In [59]:
""" test_documents = get_data_ann(df_test,nlp)
with open("test_documents.pkl", "wb") as f:
    pickle.dump(test_documents, f) """

' test_documents = get_data_ann(df_test,nlp)\nwith open("test_documents.pkl", "wb") as f:\n    pickle.dump(test_documents, f) '

In [60]:
train_documents = pd.read_pickle("train_documents.pkl")

In [61]:
val_documents = pd.read_pickle("val_documents.pkl")
test_documents = pd.read_pickle("test_documents.pkl")

### 3) configurazione della rappresentazione testuale + estrazione features dal training set

In [62]:
configurations_w = [
    ("word", 1),
    ("word", 2),
    ("word", 3),
    ("lemma", 1),
    ("lemma", 2),
    ("lemma", 3),
    ("pos", 1),
    ("pos", 2),
    ("pos", 3),
]

In [63]:
configurations_c = [3,4,5]

In [64]:
results = {}
for item, n in configurations_w:
    features_list = []
    for doc in train_documents:
        features_list.append(extract_word_ngrams(doc, item, n))
    results[(item, n)] = features_list

In [65]:
for c in configurations_c:
    features_list = []
    for doc in train_documents:
        features_list.append(extract_char_ngrams(doc, c))
    results[("char", c)] = features_list

### 4) Valutazione tramite procedura di cross-validation a 5 fold sul training set per selezionare il modello migliore

In [66]:
labels = [doc.label for doc in train_documents]

train_scores = {}

for config, features_list in results.items():
    vectorizer = DictVectorizer()
    X = vectorizer.fit_transform(features_list)
    scores = cross_val_score(LinearSVC(), X, labels, cv=5, scoring='f1')
    print(f"{config}: {scores.mean():.3f}")
    print(scores.std())
    train_scores[str(config)] = scores.mean()

('word', 1): 0.978
0.007462704962475109
('word', 2): 0.973
0.008349569580764965
('word', 3): 0.897
0.011370280318139308
('lemma', 1): 0.969
0.005848799831016869
('lemma', 2): 0.974
0.005515103752987804
('lemma', 3): 0.939
0.005687826437090686
('pos', 1): 0.923
0.011085336442628836
('pos', 2): 0.948
0.010674114167468555


c:\Users\jacop\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\jacop\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\jacop\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\jacop\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


('pos', 3): 0.950
0.006185790175668455
('char', 3): 0.968
0.008837124243243883
('char', 4): 0.971
0.004157077698008979
('char', 5): 0.978
0.008163615503032354


### 5) Estrazione feature da validation e test set per valutare le prestazioni del modello

In [67]:
labels_val = [doc.label for doc in val_documents]
labels_t = [doc.label for doc in test_documents]

In [68]:
results_val = {}
for item, n in configurations_w:
    features_list = []
    for doc in val_documents:
        features_list.append(extract_word_ngrams(doc, item, n))
    results_val[(item, n)] = features_list

In [69]:
for c in configurations_c:
    features_list = []
    for doc in val_documents:
        features_list.append(extract_char_ngrams(doc, c))
    results_val[("char", c)] = features_list

In [70]:
results_t = {}
for item, n in configurations_w:
    features_list = []
    for doc in test_documents:
        features_list.append(extract_word_ngrams(doc, item, n))
    results_t[(item, n)] = features_list

In [71]:
for c in configurations_c:
    features_list = []
    for doc in test_documents:
        features_list.append(extract_char_ngrams(doc, c))
    results_t[("char", c)] = features_list

### 6) Training del modello finale + valutazione su val e test set

In [72]:
best_config = ("word", 1)

In [73]:
best_vectorizer = DictVectorizer()
X_train_best = best_vectorizer.fit_transform(results[best_config])
best_clf = LinearSVC()
best_clf.fit(X_train_best, labels)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo rand

In [74]:
# valutazione sul validation set
X_val = best_vectorizer.transform(results_val[best_config])
print("Validation:")
print(classification_report(labels_val, best_clf.predict(X_val)))

Validation:
              precision    recall  f1-score   support

           0       0.98      0.97      0.97       500
           1       0.97      0.98      0.97       500

    accuracy                           0.97      1000
   macro avg       0.97      0.97      0.97      1000
weighted avg       0.97      0.97      0.97      1000



In [75]:
# valutazione sul test set
X_test = best_vectorizer.transform(results_t[best_config])
print("Test:")
print(classification_report(labels_t, best_clf.predict(X_test)))

Test:
              precision    recall  f1-score   support

           0       0.82      0.98      0.89       500
           1       0.97      0.78      0.87       500

    accuracy                           0.88      1000
   macro avg       0.89      0.88      0.88      1000
weighted avg       0.89      0.88      0.88      1000



### 7) Prova con trigrammi di POs 

In [76]:
new_config = ("pos",3)

In [77]:
vectorizer = DictVectorizer()
X_train = vectorizer.fit_transform(results[new_config])
clf = LinearSVC()
clf.fit(X_train, labels)

c:\Users\jacop\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo rand

In [78]:
# valutazione sul validation set
X_val = vectorizer.transform(results_val[new_config])
print("Validation:")
print(classification_report(labels_val, clf.predict(X_val)))

Validation:
              precision    recall  f1-score   support

           0       0.97      0.96      0.96       500
           1       0.96      0.97      0.96       500

    accuracy                           0.96      1000
   macro avg       0.96      0.96      0.96      1000
weighted avg       0.96      0.96      0.96      1000



In [79]:
# valutazione sul test set
X_test = vectorizer.transform(results_t[new_config])
print("Test:")
print(classification_report(labels_t, clf.predict(X_test)))

Test:
              precision    recall  f1-score   support

           0       0.77      0.94      0.85       500
           1       0.93      0.73      0.81       500

    accuracy                           0.83      1000
   macro avg       0.85      0.83      0.83      1000
weighted avg       0.85      0.83      0.83      1000



### 8) Prova con 5-grams di caratteri

In [80]:
char_config = ("char", 5)

In [81]:
vectorizer = DictVectorizer()
X_train = vectorizer.fit_transform(results[char_config])
clf = LinearSVC()
clf.fit(X_train, labels)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo rand

In [82]:
# valutazione sul validation set
X_val = vectorizer.transform(results_val[char_config])
print("Validation:")
print(classification_report(labels_val, clf.predict(X_val)))

Validation:
              precision    recall  f1-score   support

           0       0.99      0.97      0.98       500
           1       0.97      0.99      0.98       500

    accuracy                           0.98      1000
   macro avg       0.98      0.98      0.98      1000
weighted avg       0.98      0.98      0.98      1000



In [83]:
# valutazione sul test set
X_test = vectorizer.transform(results_t[char_config])
print("Test:")
print(classification_report(labels_t, clf.predict(X_test)))

Test:
              precision    recall  f1-score   support

           0       0.77      0.98      0.86       500
           1       0.97      0.71      0.82       500

    accuracy                           0.84      1000
   macro avg       0.87      0.84      0.84      1000
weighted avg       0.87      0.84      0.84      1000



### 9) Lunghezza testi per classe

Dato che gli unigrammidi parola sono conteggi grezzi, è interessante verificare se il vantaggio di questi rispetto alle altre configurazioni sperimentate, possa derivare anche da peculiarità strutturali del dataset, come la lunghezza.

In [84]:
def document_length_stats(documents, split_name):
    rows = []

    for doc in documents:
        words = doc.get_words()
        text = doc.text

        rows.append({
            "split": split_name,
            "label": doc.label,
            "n_tokens": len(words),
            "n_chars": len(text),
            "n_words_space": len(text.split())
        })

    return pd.DataFrame(rows)

df_len_train = document_length_stats(train_documents, "train")
df_len_val = document_length_stats(val_documents, "validation")
df_len_test = document_length_stats(test_documents, "test")

df_len = pd.concat([df_len_train, df_len_val, df_len_test], ignore_index=True)

In [85]:
for split in ["train", "validation", "test"]:
    temp = df_len[df_len["split"] == split]

    print(f"\n=== {split} ===")
    display(
        temp.groupby("label")[["n_tokens", "n_chars", "n_words_space"]]
        .agg(["mean", "median", "std"])
        .round(2)
    )


=== train ===


n_tokens                 n_chars                  n_words_space         \
          mean median     std     mean  median      std          mean median   
label                                                                          
0       628.13  539.0  354.22  3206.30  2787.5  1824.96        503.29  433.0   
1       312.14  273.5  180.21  1643.06  1427.0   961.39        253.22  219.0   

               
          std  
label          
0      281.79  
1      146.71


=== validation ===


n_tokens                 n_chars                  n_words_space         \
          mean median     std     mean  median      std          mean median   
label                                                                          
0       588.06  493.0  324.68  3013.08  2519.5  1671.27        471.75  396.0   
1       320.05  277.0  180.36  1689.78  1465.0   984.23        259.56  223.0   

               
          std  
label          
0      259.28  
1      147.75


=== test ===


n_tokens                  n_chars                  n_words_space         \
          mean median      std     mean  median      std          mean median   
label                                                                           
0       692.93  507.5  1420.62  3546.24  2595.0  7566.44        559.01  408.0   
1       233.39  244.0    63.63  1245.96  1297.0   326.32        190.93  198.5   

                
           std  
label           
0      1219.44  
1        50.05

In [86]:
for split in ["train", "validation", "test"]:
    temp = df_len[df_len["split"] == split]

    class_0 = temp[temp["label"] == 0]["n_tokens"]
    class_1 = temp[temp["label"] == 1]["n_tokens"]

    stat, p = mannwhitneyu(class_0, class_1, alternative="two-sided")

    print(f"\n{split}")
    print("Media classe 0:", round(class_0.mean(), 2))
    print("Media classe 1:", round(class_1.mean(), 2))
    print("Mediana classe 0:", round(class_0.median(), 2))
    print("Mediana classe 1:", round(class_1.median(), 2))
    print("Mann-Whitney U p-value:", p)


train
Media classe 0: 628.13
Media classe 1: 312.14
Mediana classe 0: 539.0
Mediana classe 1: 273.5
Mann-Whitney U p-value: 1.507107389832821e-211

validation
Media classe 0: 588.06
Media classe 1: 320.05
Mediana classe 0: 493.0
Mediana classe 1: 277.0
Mann-Whitney U p-value: 1.213306026790205e-97

test
Media classe 0: 692.93
Media classe 1: 233.39
Mediana classe 0: 507.5
Mediana classe 1: 244.0
Mann-Whitney U p-value: 1.6503180652294174e-129


### 10) Baseline con solo feature di lunghezza

In [87]:
X_train_len = df_len_train[["n_tokens", "n_chars", "n_words_space"]]
y_train_len = df_len_train["label"]

X_val_len = df_len_val[["n_tokens", "n_chars", "n_words_space"]]
y_val_len = df_len_val["label"]

X_test_len = df_len_test[["n_tokens", "n_chars", "n_words_space"]]
y_test_len = df_len_test["label"]


In [88]:
length_clf = make_pipeline(
    StandardScaler(),
    LinearSVC(max_iter=10000, random_state=42)
)

length_clf.fit(X_train_len, y_train_len)

print("Validation - solo feature di lunghezza:")
print(classification_report(y_val_len, length_clf.predict(X_val_len)))

print("Test - solo feature di lunghezza:")
print(classification_report(y_test_len, length_clf.predict(X_test_len)))

Validation - solo feature di lunghezza:
              precision    recall  f1-score   support

           0       0.86      0.64      0.73       500
           1       0.71      0.89      0.79       500

    accuracy                           0.77      1000
   macro avg       0.79      0.77      0.76      1000
weighted avg       0.79      0.77      0.76      1000

Test - solo feature di lunghezza:
              precision    recall  f1-score   support

           0       1.00      0.61      0.76       500
           1       0.72      1.00      0.84       500

    accuracy                           0.81      1000
   macro avg       0.86      0.80      0.80      1000
weighted avg       0.86      0.81      0.80      1000



### 11) Test su testo generato da chatgpt 

In [89]:
testo_1 = """
Non so bene perché, ma negli ultimi mesi ho iniziato a fare caso a dettagli che prima mi sfuggivano completamente. 
Per esempio, quando torno a casa la sera, mi accorgo subito se qualcuno ha spostato una sedia o se la luce del corridoio 
è rimasta accesa più del solito. Non è una cosa importante, anzi probabilmente non significa niente, però mi dà la sensazione 
che la giornata abbia lasciato una traccia anche prima che io riesca davvero a pensarci. Forse è solo stanchezza, oppure il fatto 
che passo troppe ore davanti al computer e poi cerco nel mondo reale qualcosa che non abbia bisogno di essere aggiornato, salvato 
o ricaricato. In ogni caso, mi sembra strano quanto certe abitudini piccole finiscano per raccontare meglio di noi rispetto alle 
cose che diciamo apertamente.
"""

In [90]:
df_manual = pd.DataFrame({
    "text": [testo_1],
    "label": [0]
})

In [91]:
manual_documents = get_data_ann(df_manual, nlp)

In [92]:
manual_config = ("word", 1)

In [93]:
if manual_config[0] == "char":
    manual_features = [extract_char_ngrams(manual_documents[0], manual_config[1])]
else:
    manual_features = [extract_word_ngrams(manual_documents[0], manual_config[0], manual_config[1])]

In [94]:
X_manual = best_vectorizer.transform(manual_features)
pred = best_clf.predict(X_manual)[0]

In [95]:
pred = best_clf.predict(X_manual)[0]

In [96]:
score = best_clf.decision_function(X_manual)[0]

In [97]:
print("Predizione:", pred)
print("Score decisionale:", score)

Predizione: 0
Score decisionale: -0.21501643100668147
